<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP9.1A/MP9.1A_u_bend.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP9.1A · One Bend of the Serpentine

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'pipe.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-cooling-plate/")
PROJECT = (['mp91a.py', 'MP9.1A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP9.1A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex09.1-cooling-plate", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex09.1-cooling-plate", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'pipe', 'mp91a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import pipe as hs
import mp91a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP9.1A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

Ex09.1 solved one straight leg of the cold plate's serpentine, **far from the bends**, where the velocity of the coolant is the same at every cross-section and Hagen–Poiseuille's parabola holds. But a serpentine is mostly bends and the legs between them. In a bend the fast liquid in the middle of the tube is thrown outwards by the centrifugal force and comes back along the walls: the fastest liquid moves towards the outer wall, and two counter-rotating vortices - **Dean vortices** - appear beside the flow along the tube. In the leg after the bend the vortices decay and the parabola forms again, over a length comparable to the entrance length of Ex09.1.

An engineer designing a cold plate wants three things from that: **the velocity across the tube through the bend and after it** (it moves the heat transfer: the vortices sweep the coolant across the hot wall), **how long the flow takes to recover** (whether the next leg starts with a parabola), and **the bend's loss coefficient $K$**, the extra pressure drop the bend costs the pump. A design study asks them for every flow rate the pump may run at, and for every bend of every candidate layout. That is where a fast model pays.

This mini project computes all three for **any flow between 0.15 and 0.45 L/min**, with one network.

## A2 · The bend, and what is new against Ex09.1

The tube, the coolant and the nominal flow are Ex09.1's: a 6 mm copper tube, 50/50 water–glycol at 40 °C, 0.3 L/min (Reynolds number 452). The domain is one U-bend of the serpentine with a piece of tube on each side:

| piece | length along the tube's axis | in tube radii $a$ = 3 mm |
|---|---|---|
| inlet leg, straight | 20 mm | 0 to 6.67 |
| U-bend, 180°, radius 12 mm to the tube's axis | 37.7 mm | 6.67 to 19.23 |
| outlet leg, straight | 110 mm | 19.23 to 55.90 |

**What is new.** Everything that makes Ex09.1 exact is gone: the velocity changes along the tube, there is a **secondary flow** across the section, and the equations are nonlinear. The strength of the change is the **curvature ratio** $\delta = a/R_c$ = 0.25 (tube radius over bend radius); at $\delta = 0$ the bend is straight and Ex09.1's parabola holds everywhere (part C checks it). The flow enters with Ex09.1's parabola: the inlet leg is far enough from the previous bend.

The cell below draws the plate (Ex09.1's drawing) and prints the data.

In [ ]:
# The serpentine of Ex09.1, and the data of this project
fig = hs.draw_domain()                                   # Ex09.1's drawing: the plate, the serpentine, the section A-A
plt.show()
rows = [("tube", f"radius a = {1e3*mp.A_TUBE:.0f} mm; coolant ρ = {hs.RHO:.0f} kg/m³, μ = {1e3*hs.MU:.2f} mPa·s (Ex09.1, typical)"),
        ("bend", f"180°, radius R_c = {1e3*mp.R_BEND:.0f} mm to the axis (assumed): δ = a/R_c = {mp.DELTA:.2f}"),
        ("legs", f"{1e3*mp.LEG_IN:.0f} mm before, {1e3*mp.LEG_OUT:.0f} mm after (assumed); the whole piece {mp.S3:.2f} tube radii"),
        ("flow", f"Q = {mp.Q_MIN} to {mp.Q_MAX} L/min, a network input: Re = {mp.reynolds(mp.Q_MIN):.0f} to {mp.reynolds(mp.Q_MAX):.0f}, "
                 f"Dean number {mp.dean_number(mp.Q_MIN):.0f} to {mp.dean_number(mp.Q_MAX):.0f}"),
        ("nominal", f"0.3 L/min: Re = {mp.reynolds(0.3):.0f}, centre speed V = {mp.centre_speed(0.3):.3f} m/s, entrance length 0.05 Re D = {1e3*hs.ENTRANCE:.0f} mm")]
for k, v in rows:
    print(f"{k:>8}  {v}")

**What you should see.** The plate with its four legs and three U-bends, and the table. The flow is laminar over the whole range (Reynolds numbers 226 to 678, below 2300), and the Dean number - the Reynolds number times $\sqrt{a/R_c}$, the measure of the bend's secondary flow - is 113 to 339: strong secondary flow, well beyond the small-Dean-number regime where Dean's series holds. The outlet leg, 110 mm, is about one entrance length (136 mm at 0.3 L/min).

## A3 · The equations and their conditions

Inside the tube the coolant obeys the Navier–Stokes equations of Ex09.1, now in three dimensions. Two approximations reduce them to what a notebook can solve, and both are stated here because the ground truth is the converged solution of **this model**, not of the full equations:

* **Dean's approximation** (small $\delta$). The tube's curvature enters only through the centrifugal force on the flow along the tube; the other curvature terms, of order $\delta$, are dropped. $\delta$ = 0.25 is at the edge of where this holds (Berger, Talbot and Yao, *Annu. Rev. Fluid Mech.* 1983, quoted from memory); the full equations would change the numbers by some per cent.
* **Parabolised** (a duct flow with a main direction). Along the tube's axis the flow is carried, not diffused: diffusion along the arc length $s$ is dropped, and nothing downstream acts upstream. The three-dimensional problem becomes a **march** of two-dimensional cross-sections from the inlet to the outlet. The secondary flow is kept divergence-free; the small cross-flow that the change of the axial velocity along the tube drives is left out.

In units of the tube radius $a$ for lengths, the straight tube's centre speed $V$ (twice the mean, as in Ex09.1) for the axial velocity $w$, and $\nu/a$ for the secondary velocity, with $x$ across the section pointing away from the bend's centre and $\mathrm{Re} = Va/\nu$ (the same number as Ex09.1's $UD/\nu$):

$$\mathrm{Re}\,w\,\frac{\partial w}{\partial s} + \mathbf{u}\cdot\nabla w = \pi(s) + \nabla^2 w \qquad\text{(momentum along the tube)}$$

$$\mathrm{Re}\,w\,\frac{\partial \omega}{\partial s} + \mathbf{u}\cdot\nabla \omega = \nabla^2 \omega - \kappa(s)\,\delta\,\mathrm{Re}^2\,\frac{\partial (w^2)}{\partial y}, \qquad \nabla^2\psi = -\omega, \qquad \mathbf{u} = \left(\frac{\partial\psi}{\partial y},\ -\frac{\partial\psi}{\partial x}\right)$$

with $\psi$ the stream function of the secondary flow, $\omega$ its vorticity, $\pi(s)$ the pressure gradient along the tube (4 in a straight tube: Poiseuille), and $\kappa(s)$ the curvature over $1/R_c$.

| where | condition | why it holds |
|---|---|---|
| inlet, $s = 0$ | $w = 1 - r^2$, $\psi = 0$ | Ex09.1's parabola: the previous bend is far enough upstream |
| the wall, $r = 1$ | $w = 0$, $\psi = 0$, $\partial\psi/\partial r = 0$ | no slip, for both the axial and the secondary flow |
| every section | $\int w\,dA = \pi/2$ | the flow rate is held: what enters leaves; $\pi(s)$ is the unknown that makes it so |
| the bend | $\kappa = 1$ from 6.67 to 19.23 radii, 0 in the legs, a smooth rise and fall over one tube radius at each end | a parabolised march carries nothing upstream; in the real tube the bend is felt about a radius ahead, and a jump in curvature would put a singular layer of vorticity on the wall |
| outlet | none | parabolised: the march needs no outlet condition |

The rise of the curvature over one tube radius is an **assumption** of this model, said again in part C.

In [ ]:
# The curvature along the tube, and what the network must return
s = np.linspace(0, mp.S3, 2000)
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(s, mp.curvature(s))
for name, sv in mp.SECTIONS.items():
    ax.axvline(sv, color="grey", ls=":", lw=0.8); ax.text(sv, 1.05, name, rotation=30, fontsize=7)
ax.set_xlabel("arc length s [tube radii]"); ax.set_ylabel("curvature κ [1/R_c]"); ax.set_ylim(-0.05, 1.3)
plt.tight_layout(); plt.show()
print("the five sections scored [tube radii from the inlet]:", ", ".join(f"{k} {v:.2f}" for k, v in mp.SECTIONS.items()))

**What you should see.** The curvature rising smoothly from 0 to 1 around 6.67 tube radii, held through the bend, falling around 19.23, and the five sections where the velocity is scored: the middle of the bend (12.95), its exit (19.23), one and five diameters after it (21.23 and 29.23), and the outlet (55.90).

## A4 · Why there is no exact solution

Ex09.1's parabola is exact because nothing changes along the tube: the convective terms vanish and the equation is linear. In the bend the axial velocity drives the secondary flow through the centrifugal term $\partial(w^2)/\partial y$, the secondary flow carries the axial velocity across the section through $\mathbf{u}\cdot\nabla w$, and both change along the tube. The equations are coupled and nonlinear in three dimensions; no closed form exists. Two limits do have reference solutions: $\delta = 0$ (Ex09.1's parabola) and a bend that never ends, whose **fully developed** flow is Dean's (`dean.py`, solved by Newton's method on the cross-section and checked against Dean's series). The serpentine's bend is only 12.6 tube radii long, so the flow leaves it before reaching that limit. The reference model and the ground truth are therefore a **numerical march** (part C), checked against both limits.

## A5 · What is asked

| quantity | at | required |
|---|---|---|
| the axial velocity at the five sections, on 441 points of each | the flows 0.20, 0.30 and 0.40 L/min, which you do not train on | within **2 % of the centre speed $V$**, root mean square over each section (the largest error reported beside it) |
| the bend's loss coefficient $K$ | the same flows | within **5 %** of the ground truth's |
| the time to answer a new flow online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline, over the whole range of flows. Its online answer - the velocity at the five sections and $K$ for a flow it has not seen - is timed against the reference model, which must march the whole tube again for every new flow. The training, and the computing of its training data, are reported apart, as the offline cost.

**Why these accuracies.** The ground truth itself is known to about 0.3 % of $V$ in root mean square, but only to about 1 % at the worst point and to about 2 % in $K$ (part C): the thin layer at the outer wall converges slowly. A requirement on the root mean square over each section is one the ground truth can judge ten times over; a requirement on the worst point would measure the ground truth's own error. The reference model, on an engineering grid, meets 2 % (part C).

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

The network is a function of four inputs and returns three numbers:

$$(s,\ x,\ y,\ Q)\ \longmapsto\ \big(w,\ \psi,\ \pi\big)$$

with $s$ the arc length in tube radii (0 to 55.90), $(x, y)$ the position in the section over the tube radius ($x$ away from the bend's centre, $x^2 + y^2 \le 1$), $Q$ the flow in L/min (0.15 to 0.45), and the outputs in the units of section A3: the axial velocity over $V(Q)$, the secondary stream function over $\nu$, and the pressure gradient along the tube ($\pi$ depends on $s$ and $Q$ only). The flow is an **input**: one training covers every flow, and a new flow is answered by evaluating the network.

The velocities are scaled by each flow's own centre speed, so the inlet is the same parabola $w = 1 - r^2$ at every flow; what the flow changes is the Reynolds number in the equations, and with it how strongly the bend deforms the profile.

## B2 · The physics the network is trained with, and the data

**The physics.** The momentum equation along the tube, section A3's first equation, is the network's residual:

$$r_\mathrm{axial} = \frac{1}{\mathrm{Re}}\Big(\mathrm{Re}\,w\,\frac{\partial w}{\partial s} + \mathbf{u}\cdot\nabla w - \pi - \nabla^2 w\Big)$$

divided by $\mathrm{Re}$ so that it is of order one at every flow. Its derivatives - first in $s$, first and second in $x$ and $y$, the secondary velocity from $\psi$ - come from autograd. The vorticity equation needs third and fourth derivatives of $\psi$ and is not used as a residual here (it is an option for you).

**The data.** A physics-informed network on this problem alone trains slowly: the secondary flow is driven by a term of order $\delta\,\mathrm{Re}^2 \approx 5\times10^4$, and the flow develops over a few tube radii from a parabola. So the network also trains on **data computed offline by the march** of part C, at ten flows that are not the test flows (0.15 to 0.45 L/min, without 0.20, 0.30 and 0.40), on a grid four times finer than the reference model's: $w$ and $\psi$ on 441 points of a section every half tube radius from the bend's start to the outlet, and $\pi$ at every step. Computing it is part of the offline cost. The loss terms:

| loss term | where it is evaluated | what it enforces |
|---|---|---|
| $\mathcal{L}_w$, $\mathcal{L}_\psi$ | the data's points | the march's axial velocity and secondary stream function ($\psi$ divided by 40, its order) |
| $\mathcal{L}_\pi$ | the data's stations | the march's pressure gradient (divided by 4) |
| $\mathcal{L}_\mathrm{axial}$ | random points $(s, x, y, Q)$ over the whole range of flows | the momentum equation along the tube, including at flows where no data exist |

**What can be built in rather than trained.**

* **The wall.** $w = (1 - r^2)\,(\ldots)$ and $\psi = (1 - r^2)^2(\ldots)$ hold no slip, and $\partial\psi/\partial r = 0$, for any weights.
* **The inlet.** Before the bend nothing changes; a factor that is zero before the curvature starts to rise and one after it keeps $w = 1 - r^2$, $\psi = 0$, $\pi = 4$ there exactly.
* **The symmetry.** The bend is symmetric about its plane: $w$ is even in $y$ and $\psi$ odd. Features $x$ and $y^2$, and a factor $y$ on $\psi$, hold it.
* **The flow rate.** $\int w\,dA = \pi/2$ at every section can be held exactly: subtract from the network's correction its flow-weighted mean over the section, computed by quadrature inside the forward pass.

`mp.sample_tube(n, seed)` draws the residual's points: $s$ from the bend's start to the outlet, half of them in the bend and the 10 radii after it, $(x, y)$ uniform over the section, $Q$ uniform over the range. The inlet leg needs no points: there the parabola is exact.

In [ ]:
# The training data: the march on 32 x 64 cells at the ten training flows (offline; a few minutes on a CPU)
DATA, PI_DATA, DATA_SECONDS = mp.training_data()          # rows (s, x, y, Q, w, ψ) and (s, Q, π)
print(f"{len(DATA)} rows of w and ψ, {len(PI_DATA)} of π, from the flows {mp.Q_TRAIN} L/min; computed in {DATA_SECONDS:.0f} s")
print("test flows in the data:", sorted(set(np.round(DATA[:, 3], 3)) & set(mp.Q_TEST)))

**What you should see.** About 437 000 rows of $w$ and $\psi$ (10 flows × 99 sections × 441 points) and 2800 values of $\pi$, computed in one to two minutes (64 to 106 s on the authors' CPU, shared with other jobs), and no test flow among them.

In [ ]:
# What the data look like: the axial velocity at the five sections, 0.35 L/min (a training flow)
x, y = mp.score_points()
fig, ax = plt.subplots(1, 5, figsize=(16, 3.2))
for a, (name, sv) in zip(ax, mp.SECTIONS.items()):
    k = np.isclose(DATA[:, 3], 0.35) & np.isclose(DATA[:, 0], mp.S1 + 0.5 * np.round((sv - mp.S1) / 0.5))
    sc = a.tricontourf(DATA[k, 1], DATA[k, 2], DATA[k, 4], levels=np.linspace(0, 1, 21), cmap="viridis")
    a.set_aspect("equal"); a.set_title(name, fontsize=9); a.set_xlabel("x (outward) [a]")
fig.colorbar(sc, ax=ax, label="w / V"); plt.show()

**What you should see.** The axial velocity in five cross-sections, the outside of the bend on the right. In the middle of the bend the fastest liquid has moved from the centre towards the outer wall, and the profile has a dent on the bend's plane where the vortices bring slow liquid from the walls inwards; at the exit it is pressed against the outer wall; one and five diameters later it is moving back towards the centre; at the outlet it is close to the parabola again.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the march an engineer would run | the same march on a much finer grid |
| grid | 16 rings × 32 cells round, steps of 0.2 tube radii | 64 rings × 128 cells round, steps of 0.1 tube radii |
| its job | to be **timed** against the network: the online cost of a new flow | to **score** the network and the reference |

Both march the same discrete equations, written in `mp91a.March`.

## C2 · The scheme, step by step

1. **The section's grid** is `dean.py`'s: rings of cells, centres at $r = (i + \tfrac12)h$, no node on the axis, the wall a face of the last ring; finite volumes for the Laplacian, central differences for the first derivatives. The fully developed bend of `dean.py` is therefore the march's own limit.
2. **Along the tube: implicit, second order.** $s$ is stepped like time with the second-order backward formula (BDF2), after one backward-Euler step at the inlet. The marching coefficient $\mathrm{Re}\,w$ vanishes at the wall: it is kept on the left and never divided by.
3. **The secondary velocity from the section before.** In each step the secondary velocity that carries $w$ and $\omega$ across the section is extrapolated from the two sections before; the step then needs only linear solves.
4. **The flow rate, exactly.** The axial equation is linear in $\pi$: it is solved twice, for $\pi = 0$ and for $\pi = 1$, and the two are added in the proportion that gives the inlet's flow rate. Every section carries exactly the flow fed.
5. **The vorticity and the stream function together**, one sparse solve, with the wall's vorticity from Thom's condition (from the stream function half a cell inside the wall, as `dean.py`).
6. **The curvature** rises from 0 to 1 over one tube radius centred on the bend's start and falls the same way at its end (section A3).
7. **The loss coefficient.** $K$ is the pressure drop over the whole piece in excess of a straight tube of the same length, over $\rho\bar u^2/2$: $K = (8/\mathrm{Re})\int(\pi - 4)\,ds$, the integral by the trapezoid rule over the steps.

`mp.March(n, m, ds).run(Q, keep=...)` marches at a flow; `mp.reference(Q)` runs it on the engineering grid and returns the five sections on the score points and $K$.

In [ ]:
# The ground truth: its convergence study at 0.3 L/min, and its checks, from the truth file
G = np.load("MP9.1A_truth.npz")
print("across the section (steps of 0.1 radii):")
for g, K, wm, sec in zip(G["conv_grids"], G["conv_K"], G["conv_wmax_exit"], G["conv_seconds"]):
    print(f"   {int(g[0]):3d} x {int(g[1]):3d} cells: K = {K:.4f}, fastest liquid at the bend exit {wm:.4f} V   ({sec:.0f} s)")
for i, (dm, dr) in enumerate(zip(G["conv_change"], G["conv_change_rms"]), 1):
    print(f"   change of w at the five sections, grid {i} to {i+1}: largest {dm:.2e} V, root mean square {dr:.2e} V")
print("along the tube (32 x 64 cells):")
for g, K in zip(G["step_grids"], G["step_K"]):
    print(f"   steps of {g[2]:.2f} radii: K = {K:.4f}")
for i, (dm, dr) in enumerate(zip(G["step_change"], G["step_change_rms"]), 1):
    print(f"   change of w, step {i} to {i+1}: largest {dm:.2e} V, root mean square {dr:.2e} V")
print(f"   K: observed order {float(G['step_K_order']):.2f}, Richardson extrapolation {float(G['step_K_extrapolated']):.4f}")
z = G["zero_strength"]; lb = G["long_bend"]
print(f"delta = 0: the march keeps the grid's parabola to {z[0]:.1e} V (the 16-ring grid's parabola is within {z[1]:.1e} V of 1 - r²)")
print(f"a bend of 300 radii: friction over the straight tube's {lb[0]:.4f}, dean.py's fully developed bend {lb[1]:.4f}; "
      f"axial velocity off by {lb[2]:.1e} V, secondary flow by {lb[3]:.1e} of its largest value")
for q in mp.Q_TEST:
    sym = G[f"symmetry_{q:.2f}"]
    print(f"{q:.2f} L/min: K = {float(G[f'K_{q:.2f}']):.4f}; symmetry about the bend's plane held to {sym[0]:.0e} (w) and {sym[1]:.0e} (ψ)")

**What you should see.** *Across the section*, the largest change of the axial velocity at the five sections falls from 3.1e-02 V (16 to 32 rings) to 1.1e-02 V (32 to 64): a factor 2.7, an observed order of about 1.4, not the 2 of the interior scheme, because the layer at the outer wall, where the fast liquid is pressed, is thin and the wall vorticity is first order. In root mean square the change falls from 7.2e-03 to 3.4e-03 V: **the 64-ring ground truth is within about 0.3 % of V in root mean square, about 1 % at the worst point.** $K$ changes by 1.5 % and then 1.9 % (1.2200, 1.2021, 1.1793): its changes do not fall, so it has no observed order and is not extrapolated; its uncertainty is taken as the last change, **about 2 %**. *Along the tube* the march is cleanly second order: $K$ = 1.1838, 1.1982, 1.2021, 1.2030 for steps of 0.4 to 0.05 radii (observed order 2.13), the velocity changing by 2.4e-03 V between the last two; the ground truth's 0.1 radii are within 0.1 % of the extrapolated $K$.

*The checks.* At $\delta = 0$ the march keeps the parabola to rounding (1.6e-14), and the grid's parabola is within 1e-03 V of Hagen–Poiseuille's (the grid's own error, 16 rings). In a bend made 300 radii long the march reaches `dean.py`'s fully developed bend on the same grid: friction 2.0152 times the straight tube's in both, velocity within 2.4e-08 V. The symmetry about the bend's plane holds to rounding, and the flow rate is held exactly at every section. The ground truth is a converged solution **of the parabolised Dean model**; the model's own approximations (section A3) are not measured here.

In [ ]:
# The zero-strength check, run here: delta = 0, the march must keep Ex09.1's parabola at every section
mp.check_zero_strength();

**What you should see.** The outlet against the grid's parabola to about 1e-14 V, the pressure gradient 4 (Poiseuille) to 1e-13, no secondary flow: with the bend straightened the march is Ex09.1's steady limit.

## C3 · The reference model: accuracy and time

The reference grid, 16 × 32 cells with steps of 0.2 radii, is the coarsest tried that meets 2 % of $V$ in root mean square at every test flow (12 × 24 cells misses at 0.40 L/min, 2.9 %). The cell below scores it against the ground truth and times it. It runs on the CPU: a march of 280 sparse solves gains nothing from a GPU, and an engineer would not run it on one.

In [ ]:
# The reference model against the ground truth, at the three test flows
x, y = G["x"], G["y"]
worst_rms, worst_K = 0.0, 0.0
for q in mp.Q_TEST:
    r = mp.reference(q)
    e = {k: r["sections"][k][0] - G[f"w_{q:.2f}_{k.replace(' ', '_')}"] for k in mp.SECTIONS}
    rms = max(mp.rms(v) for v in e.values()); mx = max(np.abs(v).max() for v in e.values())
    eK = abs(r["K"] / float(G[f"K_{q:.2f}"]) - 1)
    worst_rms, worst_K = max(worst_rms, rms), max(worst_K, eK)
    print(f"{q:.2f} L/min: axial velocity off by {100*rms:.2f} % of V (root mean square, worst section), {100*mx:.2f} % at the worst point; "
          f"K = {r['K']:.4f} (truth {float(G[f'K_{q:.2f}']):.4f}, {100*eK:.1f} %); {r['seconds']:.2f} s")
t_ref = middle(lambda: mp.reference(0.27), reps=3)        # the online cost of one new flow
print(f"\nreference model: {t_ref:.2f} s per new flow; the network must answer in under {t_ref*1e2:.0f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "worst_rms": worst_rms, "worst_K": worst_K}   # kept for part E

**What you should see.** The reference model's axial velocity within 0.80, 1.03 and 1.60 % of $V$ in root mean square at 0.20, 0.30 and 0.40 L/min - inside the 2 % agreed - and 2.2 to 5.8 % at the worst point, in the thin layer at the outer wall; its $K$ within 3.7, 3.1 and 1.2 %, inside 5 %. One march takes 1.3 to 2.1 s on the authors' CPU, shared with other jobs (280 steps, each two sparse factorisations). That is the bar for the network's online time, which part E measures on this same runtime.

In [ ]:
# The ground truth at 0.30 L/min: the axial velocity and the secondary flow at the five sections
fig, ax = plt.subplots(2, 5, figsize=(16, 6))
for i, name in enumerate(mp.SECTIONS):
    tag = name.replace(" ", "_")
    c = ax[0, i].tricontourf(x, y, G[f"w_0.30_{tag}"], levels=np.linspace(0, 1, 21), cmap="viridis")
    ax[0, i].set_title(name, fontsize=9)
    ax[1, i].tricontour(x, y, G[f"psi_0.30_{tag}"], levels=15, cmap="coolwarm", linewidths=0.8)
    for a in ax[:, i]:
        a.add_patch(plt.Circle((0, 0), 1, fill=False, color="k", lw=0.6)); a.set_aspect("equal"); a.axis("off")
fig.colorbar(c, ax=ax[0, :], label="w / V"); ax[1, 0].set_title("ψ: the secondary flow", fontsize=9, loc="left")
plt.show()
plt.figure(figsize=(9, 2.8))
for q in mp.Q_TEST:
    plt.plot(G["s"], G[f"pi_{q:.2f}"], label=f"{q:.2f} L/min, K = {float(G[f'K_{q:.2f}']):.3f}")
plt.axhline(4, color="grey", lw=0.6); plt.xlabel("s [tube radii]"); plt.ylabel("pressure gradient π (4 = straight)"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

**What you should see.** Top, the axial velocity: pushed to the outer wall (right) through the bend, slowly returning after it. Bottom, the secondary flow's streamlines: two vortices, symmetric about the bend's plane, strongest in the bend and decaying after it. Below, the pressure gradient along the tube: 4 in the inlet leg, rising to about 8 in the bend - the friction of a fully developed bend is about twice the straight tube's - and decaying back to 4 over the outlet leg. The area between the curve and 4 is the bend's loss: $K$ = 1.27, 1.18 and 1.15 at 0.20, 0.30 and 0.40 L/min.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given, so that your month goes into the deep learning.

**The contract.** Your network is a PyTorch module called `net`, used as `w, psi, pi = net(s, x, y, Q)`:

| argument | meaning | shape |
|---|---|---|
| `s` | arc length in tube radii, 0 to 55.90 | `(n, 1)` |
| `x`, `y` | position in the section over the tube radius, $x$ away from the bend's centre | `(n, 1)` |
| `Q` | the flow in L/min, 0.15 to 0.45 | `(n, 1)` |
| returns | $w/V$, $\psi/\nu$, and the pressure gradient $\pi$ (section A3) | three `(n, 1)` |

## D2 · The loss, given

`data_loss(net, i, j)` compares the network with the rows `i` of `DATA` and `j` of `PI_DATA` (section B2): $w$, $\psi/40$ and $\pi/4$, squared and averaged. `axial_loss(net, n, seed)` is the momentum equation along the tube, divided by Re, at `n` points drawn by `mp.sample_tube`.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a, grad=False):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1).requires_grad_(grad)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

DATA_T = [T(DATA[:, k]) for k in range(6)]                # s, x, y, Q, w, ψ on the device
PI_T = [T(PI_DATA[:, k]) for k in range(3)]               # s, Q, π

def data_loss(net, i, j):
    s, x, y, Q, w, psi = (v[i] for v in DATA_T)
    w_n, psi_n, _ = net(s, x, y, Q)
    sp_, Qp, pi = (v[j] for v in PI_T)
    _, _, pi_n = net(sp_, torch.zeros_like(sp_), torch.zeros_like(sp_), Qp)   # π depends on s and Q only
    return {"w": (w_n - w).pow(2).mean(), "psi": ((psi_n - psi) / 40).pow(2).mean(), "pi": ((pi_n - pi) / 4).pow(2).mean()}

def axial_loss(net, n=256, seed=0):
    # Re w w_s + u.grad w = π + lap w, divided by Re, at n random points (s, x, y, Q)
    p = mp.sample_tube(n, seed)
    s, x, y = T(p[:, 0], True), T(p[:, 1], True), T(p[:, 2], True)
    Q = T(p[:, 3]); Re = T(mp.reynolds(p[:, 3]))
    w, psi, pi = net(s, x, y, Q)
    wx, wy = d(w, x), d(w, y)
    u, v = d(psi, y), -d(psi, x)                         # the secondary velocity from the stream function
    r = (Re * w * d(w, s) + u * wx + v * wy - pi - d(wx, x) - d(wy, y)) / Re
    return r.pow(2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The four inputs scaled to order one, or features of them: of $s$, functions that follow the bend and the decay after it (the secondary flow decays over some tube radii); of the section, $x$ and $y^2$ (symmetry);
* **what is built in** (section B2): the wall, the inlet, the symmetry, the flow rate;
* **its size and form.** Layers, neurons, activation. The velocity in a section is a combination of a few shapes whose amplitudes change along the tube and with the flow: a network for the shapes and one for their amplitudes, multiplied, is one way to use that.

What the problem tells you, from parts A to C: the axial velocity changes by up to 0.5 V across the section within a few tube radii, and the root mean square asked is 2 % of V.

In [ ]:
# TODO 1 - your network. net(s, x, y, Q) -> (w, psi, pi), every argument an (n, 1) tensor (see the table above)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, s, x, y, Q):
        ...                                              # return w, psi, pi, each of shape (n, 1)

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `data_loss` and `axial_loss`. Decide the optimiser or optimisers, the number of steps, the learning rate and its schedule, the weight of each term, the batch size and whether it is drawn afresh. Keep the history of each term for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost (the data's `DATA_SECONDS` is added to it).

The course's recipe (Adam on fresh batches, then L-BFGS on one fixed batch) is a good start. `quick_score()` below gives, at the three test flows, the worst root mean square error of the axial velocity over the five sections, in per cent of V, and the worst error of $K$, in per cent.

In [ ]:
# A quick score, given: at the test flows, the worst section's root mean square error of w [% of V] and the worst error of K [%]
G = np.load("MP9.1A_truth.npz")
S_LINE = np.linspace(0, mp.S3, 2001)                     # stations for K
def net_K(Q):
    with torch.no_grad():
        _, _, pi = net(T(S_LINE), T(np.zeros_like(S_LINE)), T(np.zeros_like(S_LINE)), T(np.full_like(S_LINE, Q)))
    return mp.k_from_pi(pi.cpu().numpy().ravel(), S_LINE, mp.reynolds(Q))

def quick_score():
    worst_w, worst_K = 0.0, 0.0
    x, y = G["x"], G["y"]
    for Q in mp.Q_TEST:
        for name, sv in mp.SECTIONS.items():
            with torch.no_grad():
                w, _, _ = net(T(np.full_like(x, sv)), T(x), T(y), T(np.full_like(x, Q)))
            worst_w = max(worst_w, mp.rms(w.cpu().numpy().ravel() - G[f"w_{Q:.2f}_{name.replace(' ', '_')}"]))
        worst_K = max(worst_K, abs(net_K(Q) / float(G[f"K_{Q:.2f}"]) - 1))
    return 100 * worst_w, 100 * worst_K

In [ ]:
# TODO 2 - your training
history = {"w": [], "psi": [], "pi": [], "axial": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
ew, eK = quick_score()
print(f"trained in {TRAIN_SECONDS:.0f} s; axial velocity off by {ew:.2f} % of V (root mean square, worst section), K by {eK:.1f} %")

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the axial velocity at one section and one test flow against the ground truth, with its error. Save each figure you want in the report as `MP9.1A_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* $w$ and $\psi$ at the five sections on the ground truth's 441 points, at the test flows 0.20, 0.30 and 0.40 L/min, and at 0.10 and 0.50 L/min outside the range;
* $K$ at eleven flows from 0.15 to 0.45 L/min and at the test and outside flows;
* the largest $|\psi|$ along the tube at the test flows (for the decay of the secondary flow);
* the **online time** of a new flow on this runtime - the five sections and $K$ from one call of `net` - and the same for 10 and 100 flows at once, against the reference model's march (its time from part C, `REF`).

Set `TRAIN_SECONDS` to the offline training time you measured; it is reported beside the online times.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
x, y = G["x"], G["y"]
NS = len(mp.SECTIONS)
SV = np.array(list(mp.SECTIONS.values()))
def answer(flows):
    # the five sections (w, ψ on the 441 points) and K for several flows, from one call of the network
    flows = np.atleast_1d(np.asarray(flows, dtype=float)); n, m = len(flows), x.size
    s = np.r_[np.tile(np.repeat(SV, m), n), np.tile(S_LINE, n)]
    xa = np.r_[np.tile(x, NS * n), np.zeros(S_LINE.size * n)]
    ya = np.r_[np.tile(y, NS * n), np.zeros(S_LINE.size * n)]
    Qa = np.r_[np.repeat(flows, NS * m), np.repeat(flows, S_LINE.size)]
    with torch.no_grad():
        w, psi, pi = net(T(s), T(xa), T(ya), T(Qa))
    k = NS * m * n
    w = w[:k].reshape(n, NS, m).cpu().numpy(); psi = psi[:k].reshape(n, NS, m).cpu().numpy()
    pi = pi[k:].reshape(n, S_LINE.size).cpu().numpy()
    K = np.array([mp.k_from_pi(p, S_LINE, mp.reynolds(Q)) for p, Q in zip(pi, flows)])
    return w, psi, K, pi

flows = {"test": np.array(mp.Q_TEST), "outside": np.array([0.10, 0.50]), "sweep": np.linspace(0.15, 0.45, 11)}
out = {"x": x, "y": y, "sections": np.array(list(mp.SECTIONS)), "s_line": S_LINE}
for key, f in flows.items():
    out[f"w_{key}"], out[f"psi_{key}"], out[f"K_{key}"], out[f"pi_{key}"] = answer(f)
    out[f"flows_{key}"] = f

# the strength of the secondary flow along the tube: the largest |ψ| over a section, every half radius
s_dec = np.arange(mp.S1, mp.S3 + 1e-9, 0.5); xs, ys = mp.score_points(15)
for Q in mp.Q_TEST:
    with torch.no_grad():
        _, psi, _ = net(T(np.repeat(s_dec, xs.size)), T(np.tile(xs, s_dec.size)), T(np.tile(ys, s_dec.size)), T(np.full(s_dec.size * xs.size, Q)))
    out[f"psi_max_{Q:.2f}"] = np.abs(psi.reshape(s_dec.size, xs.size).cpu().numpy()).max(1)
out["s_decay"] = s_dec

# online time: the network against the reference model, on this runtime
timing = {}
for k in (1, 10, 100):
    fs = np.linspace(0.151, 0.449, k)
    timing[f"net_{k}"] = middle(lambda: answer(fs))
timing["ref_1"] = REF["t_ref_s"]
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS); out["data_seconds"] = np.array(DATA_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP9.1A_net.pt"))   # the trained weights, beside the report
for k in (1, 10, 100):
    print(f"online, {k:3d} new flow(s): network {timing[f'net_{k}']*1e3:7.2f} ms")
print(f"reference model, one new flow: {REF['t_ref_s']*1e3:.0f} ms ({REF['t_ref_s']/timing['net_1']:.0f} times the network's)")
for Q, K in zip(mp.Q_TEST, out["K_test"]):
    print(f"{Q:.2f} L/min: K = {K:.4f} (truth {float(G[f'K_{Q:.2f}']):.4f})")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's results `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP9.1A_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): {100*REF['worst_rms']:.2f} % of V, K within {100*REF['worst_K']:.1f} %, {REF['t_ref_s']:.2f} s per new flow")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* at each test flow: the worst section's root mean square error of the axial velocity, and the largest error at any point, both in per cent of V;
* at each test flow: the error of $K$, in per cent;
* the network's online time for one new flow, the reference model's on the same runtime, and their ratio;
* the offline cost: the data's computing time and the training time.

Then say in one line each whether the three requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>55}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP9.1A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the axial velocity at the five sections at one test flow, network against ground truth, with the error; $K$ against the flow (`R["flows_sweep"]`, `R["K_sweep"]`) with the ground truth's three points; the strength of the secondary flow along the tube (`R["psi_max_0.30"]` on `R["s_decay"]`); and the online times against the reference model's.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 6 are about your network and its result, 7 to 9 ask you to use it, 10 concludes. The next cell writes `MP9.1A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs and features, what is built in (wall, inlet, symmetry, flow rate), its form, layers, neurons and number of weights. Why this design?": """
""",
    "2. Your loss: the weight of each term, and each term's value at the end of training. What did the momentum residual change, against training on the data alone (same seed and steps)?": """
""",
    "3. Your training: optimiser(s), steps, learning rates, batch sizes, the offline time on the T4 in seconds, and the time to compute the training data.": """
""",
    "4. At 0.20, 0.30 and 0.40 L/min: the worst section's root mean square error and the largest point error of the axial velocity, in per cent of V. Is the 2 % met at every flow? Where in the section is your error largest, and why there?": """
""",
    "5. The loss coefficient K at the three test flows: your values against the ground truth's, in per cent, and against the reference model's. Is the 5 % met? How does the ground truth's own uncertainty in K (part C) compare?": """
""",
    "6. The online time for a new flow: your network against the reference model on the same runtime, in milliseconds, and the ratio; the same for 10 and 100 flows at once. What does each method spend its time on?": """
""",
    "7. The secondary flow after the bend: from your network, the distance after the bend's exit (in tube radii and in mm) at which the largest |ψ| has fallen to a tenth of its value at the exit, at 0.20, 0.30 and 0.40 L/min. How does it compare with Ex09.1's entrance length 0.05 Re D?": """
""",
    "8. Outside the training range: your errors at 0.10 and 0.50 L/min against the march on 32 x 64 cells (mp.reference(Q, grid=mp.DATA_GRID)). How fast does the error grow outside the range, and why?": """
""",
    "9. Train again with another seed. How much do the worst root mean square error and K at the three test flows differ between the two runs? Is your result reproducible within the requirements?": """
""",
    "10. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time (training and data)? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" if abs(v) >= 0.01 else f"| {k} | {v:.2e} |" for k, v in METRICS.items()]
    out = ["# MP9.1A — One Bend of the Serpentine", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP9.1A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP9.1A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP9.1A_report.md into MP9.1A_report.pdf, with any figure
# saved as MP9.1A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP9.1A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP9.1A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP9.1A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP9.1A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass